<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

Each stage has **two paths**:
- **🚧 Compute**: Run from scratch (for debugging a single episode)
- **☁️ Load**: Skip computation, load pre-computed results from GCS (for debugging later stages)

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `gs://dm-tapnet/mv-tap/droid/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `gs://dm-tapnet/mv-tap/droid/extrinsics/` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `gs://dm-tapnet/mv-tap/droid/tracks/` | Dense 3D point tracks |

---
## 0. Environment Setup

In [ ]:
# @title 0a. Clone repo & install dependencies
import os

REPO_DIR = "/content/droid"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/yangyi02/droid.git {REPO_DIR}
else:
    print(f"⏭️ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

%cd {REPO_DIR}
!bash setup.sh

In [ ]:
# @title 0b. Python imports & sys.path setup
import sys, os, json, random
import numpy as np
import torch
import mediapy as media

REPO_DIR = "/content/droid"
for p in [
    REPO_DIR,
    os.path.join(REPO_DIR, "third_party/s2m2/src"),
    os.path.join(REPO_DIR, "third_party/co-tracker"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)
os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# @title 🔄 Dev: Sync from GitHub + Hot Reload (run after pushing changes)
import importlib, subprocess

result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

import core.geometry, core.io, core.depth, core.physics, core.tracking
for mod in [core.geometry, core.io, core.depth, core.physics, core.tracking]:
    importlib.reload(mod)

import compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks
for mod in [compute_depth, compute_extrinsics, compute_tracks, compute_2d_tracks]:
    importlib.reload(mod)

import utils.visualization
importlib.reload(utils.visualization)
print("✅ All modules reloaded. Re-run cells below to test changes.")

In [ ]:
# @title 0c. Download DROID metadata (shared by all stages)

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"
files = ["camera_serials.json", "episode_id_to_path.json",
         "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)
for f in files:
    os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")

def load_json(name):
    with open(os.path.join(root_path, name)) as f:
        return json.load(f)

serials_db = load_json(files[0])
id_to_path = load_json(files[1])
keep_ranges = load_json(files[2])
extrinsics_db = load_json(files[3])

# Option 1: From metadata intersection (uncomment)
# valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()) & set(extrinsics_db.keys()))

# Option 2: From episodes_success.txt
with open("episodes_success.txt") as f:
    valid_ids = sorted([line.strip() for line in f if line.strip()])

print(f"✅ Metadata ready: {len(valid_ids)} episodes")

In [ ]:
# @title 0d. Select episode

# Option 1: Random
episode_id = random.choice(valid_ids)

# Option 2: Manual override (uncomment)
# episode_id = "ILIAD+5e938e3b+2023-07-20-11h-50m-51s"

print(f"🎯 Episode: {episode_id}")

In [ ]:
# @title 0e. Initialize scene_constants (lightweight, no SVO)
from compute_depth import init_episode

scene_constants = init_episode(
    episode_id,
    os.path.expanduser("~/droid_data/input/robotics/droid_raw/1.0.1"),
    id_to_path, serials_db, keep_ranges)
print(f"✅ scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Stage 1: Depth

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 1A. 🚧 COMPUTE depth from scratch (SVO decode + S2M2 + SAM)
# This cell installs ZED SDK + runs the full depth pipeline.
# Skip this entirely if using 1B (Load from GCS).

# --- Install ZED SDK (only runs once) ---
import shutil
if not shutil.which("ZED_Explorer"):
    !apt-get update -qq
    !apt-get install -y zstd
    sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
    !wget -q -O {sdk_installer} https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22
    !chmod +x {sdk_installer}
    !./{sdk_installer} silent runtime_only skip_tools
    !find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;
    print("✅ ZED SDK installed")
else:
    print("⏭️ ZED SDK already installed")
import pyzed.sl as sl

# --- Run depth pipeline ---
from compute_depth import (
    init_all_models, extract_svo_video,
    parse_robot_kinematics, align_temporal_streams, export_to_disk,
)
from core.depth import (
    compute_stereo_depth, build_universal_gripper_mask,
    distill_empirical_gripper_depth, inject_gripper_depth,
)

# Init models (only first time)
if 's2m2_model' not in dir():
    s2m2_model, run_stereo_matching, sam_predictor = init_all_models()

scene_constants = extract_svo_video(scene_constants)
scene_constants = parse_robot_kinematics(scene_constants)
scene_constants = align_temporal_streams(scene_constants)
scene_constants = compute_stereo_depth(
    scene_constants, s2m2_model, run_stereo_matching, device)

# Gripper refinement
wrist_serial = scene_constants["meta"].get("wrist_serial")
if wrist_serial and wrist_serial in scene_constants["camera"]:
    wrist_data = scene_constants["camera"][wrist_serial]
    if "raw_depth" in wrist_data:
        wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()
scene_constants = build_universal_gripper_mask(scene_constants, sam_predictor)
scene_constants = distill_empirical_gripper_depth(scene_constants)
scene_constants = inject_gripper_depth(scene_constants)

export_to_disk(scene_constants)
print("✅ Stage 1 COMPUTE complete")

In [ ]:
# @title 1B. ☁️ LOAD depth from GCS bucket (skip Stage 1 computation)
# Run this if depth was already computed by run_parallel.sh.

GCS_DEPTH = "gs://dm-tapnet/mv-tap/droid/depth"
local_cache = f"/content/droid_depth_cache/{episode_id}"
os.makedirs(local_cache, exist_ok=True)

# Download robot data
os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{local_cache}/' > /dev/null 2>&1")
robot_data = np.load(f"{local_cache}/robot.npz", allow_pickle=True)
for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
    if k in robot_data:
        scene_constants['robot'][k] = robot_data[k]
if 'valid_indices' in robot_data:
    scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
if 'wrist_serial' in robot_data:
    scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())
wrist_serial = scene_constants['meta'].get('wrist_serial')
print(f"  ✅ robot.npz loaded")

# Download per-camera data
base_files = ["video_left.mp4", "video_right.mp4",
              "video_left_raw.mp4", "video_right_raw.mp4",
              "raw_depth.npz", "calibration.npz"]

for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    cam_files = list(base_files)
    if cam == wrist_serial:
        cam_files.extend(["original_raw_depth.npz", "gripper_mask.npz", "gripper_depth.npz"])

    gcs_files = " ".join([f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in cam_files])
    os.system(f"gsutil -m cp {gcs_files} '{cam_dir}/' > /dev/null 2>&1")

    # Videos
    for mem_key, fname in [("video_rgb", "video_left.mp4"), ("video_right", "video_right.mp4"),
                           ("video_raw_rgb", "video_left_raw.mp4"), ("video_raw_right", "video_right_raw.mp4")]:
        vid_path = os.path.join(cam_dir, fname)
        if os.path.exists(vid_path):
            scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

    # Depth
    depth_path = os.path.join(cam_dir, "raw_depth.npz")
    if os.path.exists(depth_path):
        scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

    # Wrist extras
    for npz_key, mem_key, is_depth in [
        ("original_raw_depth.npz", "original_raw_depth", True),
        ("gripper_mask.npz", "sam_real_masks", False),
        ("gripper_depth.npz", "empirical_gripper_depth", True)]:
        p = os.path.join(cam_dir, npz_key)
        if os.path.exists(p):
            d = np.load(p)
            key = 'depth' if 'depth' in d else 'mask'
            val = d[key]
            if is_depth:
                val = val.astype(np.float32) / 1000.0
            scene_constants['camera'][cam][mem_key] = val

    # Calibration
    calib_path = os.path.join(cam_dir, "calibration.npz")
    if os.path.exists(calib_path):
        c = np.load(calib_path)
        scene_constants['camera'][cam]['K_mat'] = c['K_calib_left']
        scene_constants['camera'][cam]['baseline'] = float(c['baseline'])
        scene_constants['camera'][cam]['zed_calibration'] = {
            'calibrated': {'K': c['K_calib_left'], 'disto': c['disto_calib_left'],
                           'K_right': c['K_calib_right'], 'disto_right': c['disto_calib_right']},
            'raw': {'K': c['K_raw_left'], 'disto': c['disto_raw_left'],
                    'K_right': c['K_raw_right'], 'disto_right': c['disto_raw_right']},
        }
    print(f"  ✅ Camera {cam} loaded")

print(f"✅ Stage 1 LOADED from GCS")

In [ ]:
# @title 1. Visualize depth results
from utils.visualization import inspect_dict_structure, render_multicam_disparity_video

inspect_dict_structure(scene_constants)

frames = render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

---
## 1.5 Per-View 2D Point Tracking

Run **CoTracker** and **TAPNext** on all cameras.
Populates `tracking_results` for downstream track-reprojection metrics in Stage 2.

In [ ]:
# @title 1.5a. Run 2D tracking (CoTracker + TAPNext)
import importlib, compute_2d_tracks
importlib.reload(compute_2d_tracks)
from compute_2d_tracks import init_tracker, run_2d_tracking

GRID_SIZE = 30  # @param {type:"integer"}

tracking_results = {}
for method in ("cotracker", "tapnext"):
    _cache_key = f"_tracker_{method}"
    if _cache_key not in dir():
        globals()[_cache_key] = init_tracker(method, device)
    scene_constants = run_2d_tracking(
        globals()[_cache_key], scene_constants, device, grid_size=GRID_SIZE)
    tracking_results[method] = {
        cam_id: {
            'tracks_2d': scene_constants['camera'][cam_id]['tracks_2d'].copy(),
            'vis_2d': scene_constants['camera'][cam_id]['vis_2d'].copy(),
        }
        for cam_id in scene_constants['camera']
        if 'tracks_2d' in scene_constants['camera'][cam_id]
    }

print(f"✅ tracking_results ready: {list(tracking_results.keys())}")

In [ ]:
# @title 1.5b. Tracking video (select method)
import importlib, utils.visualization
importlib.reload(utils.visualization)
from utils.visualization import render_2d_tracking_video

VIS_METHOD = "cotracker"  # @param ["cotracker", "tapnext"]

if VIS_METHOD not in tracking_results:
    print(f"⚠️ Method '{VIS_METHOD}' not in tracking_results. "
          f"Available: {list(tracking_results.keys())}")
else:
    all_frames = []
    for cam_id in scene_constants['camera']:
        cam_data = scene_constants['camera'][cam_id]
        if cam_id not in tracking_results[VIS_METHOD]:
            continue
        tracks = tracking_results[VIS_METHOD][cam_id]['tracks_2d']
        vis = tracking_results[VIS_METHOD][cam_id]['vis_2d']
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        all_frames.append(np.array(frames))

    if all_frames:
        combined = np.concatenate(all_frames, axis=2)
        media.show_video(combined, fps=10,
                         title=f"2D Tracks [{VIS_METHOD}] — All Cameras")

---
## 2. Stage 2: Extrinsics

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 2A. 🚧 COMPUTE extrinsics from scratch (VGGT + robot alignment)

from compute_extrinsics import (
    init_extrinsics,
    run_stage2_alignment, run_global_joint_alignment,
    evaluate_extrinsics, print_metrics, prepare_track_anchors,
    export_extrinsics,
)
from core.physics import PyBulletRenderer

# Init renderers (only first time)
if 'tensor_renderer' not in dir():
    from core.physics import TensorRobotRenderer
    tensor_renderer = TensorRobotRenderer(device=device)
if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

def _eval_and_print(scene_state, stage_name):
    """Evaluate metrics + dual-tracker track reproj in one shot."""
    base = evaluate_extrinsics(scene_constants, scene_state, device,
                               tensor_renderer=tensor_renderer)
    print_metrics(base, stage_name)
    for method, tr in tracking_results.items():
        try:
            for cid, d in tr.items():
                scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
                scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
            anchors = prepare_track_anchors(
                scene_constants, scene_state, pb_renderer, device)
            m = evaluate_extrinsics(scene_constants, scene_state, device,
                                    tensor_renderer=tensor_renderer,
                                    track_anchors=anchors)
            print(f"  🎯 Track reproj ({method:>10s}): "
                  f"{m['track_reproj_mean_px']:.2f} px")
        except Exception as e:
            print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

# Stage 0+1: Dataset extrinsics → VGGT fallback
_vggt = globals().get('_vggt_models', None)
scene_state, _vggt_models = init_extrinsics(
    scene_constants, extrinsics_db, device, vggt_models=_vggt)
_eval_and_print(scene_state, "Stage 0+1 (Init)")

# Stage 2: Unified camera-robot alignment
scene_state = run_stage2_alignment(
    scene_constants, tensor_renderer, scene_state)
_eval_and_print(scene_state, "Stage 2 (Per-Camera)")

# Stage 3: Global joint optimization
scene_state = run_global_joint_alignment(
    scene_constants, scene_state, tensor_renderer)
_eval_and_print(scene_state, "Stage 3 (Global Joint)")

export_extrinsics(scene_constants, scene_state)
print("✅ Stage 2 COMPUTE complete")

In [ ]:
# @title 🔍 DEBUG: Track Reprojection Error Breakdown
# Decomposes the track reproj loss per-camera, per-frame, per-track
# to find WHERE the error comes from.

import torch, numpy as np, matplotlib.pyplot as plt
from compute_extrinsics import prepare_track_anchors, compute_track_reproj_loss
from compute_2d_tracks import init_tracker, run_2d_tracking
import importlib, compute_extrinsics; importlib.reload(compute_extrinsics)
from compute_extrinsics import prepare_track_anchors

# ── 1. Run tracking ──
tracker_name = "tapnext"  # or "cotracker"
if 'dbg_tracker' not in dir() or dbg_tracker.name.lower() != tracker_name:
    dbg_tracker = init_tracker(tracker_name, device)

import copy
sc_dbg = copy.deepcopy(scene_constants)
sc_dbg = run_2d_tracking(dbg_tracker, sc_dbg, device, grid_size=30)

# ── 2. Prepare anchors ──
anchors = prepare_track_anchors(sc_dbg, scene_state, pb_renderer, device)

# ── 3. Per-camera detailed breakdown ──
T_ee_all = scene_constants['robot']['T_ee_base_all']
T_ee_t = torch.tensor(T_ee_all, dtype=torch.float32, device=device)
wrist_cam = scene_constants['meta']['wrist_serial']

for cam_id, anchor in anchors.items():
    T_opt = torch.tensor(
        scene_state[cam_id]['base_extrinsic'],
        dtype=torch.float32, device=device)
    K = torch.tensor(
        scene_constants['camera'][cam_id]['K_mat'],
        dtype=torch.float32, device=device)
    scheme = anchor['scheme']

    P_cam0 = anchor['P_cam0']       # (N, 4)
    targets = anchor['tracks_2d']   # (T, N, 2)
    vis = anchor['vis']             # (T, N)

    # ── Reproject (same logic as compute_track_reproj_loss) ──
    if scheme in ("robot", "gripper"):
        T_base_to_cam = torch.linalg.inv(T_opt)
        T_ee_0_inv = torch.linalg.inv(T_ee_t[0])
        P_world0 = T_opt @ P_cam0.T
        P_ee = T_ee_0_inv @ P_world0
        P_world_all = T_ee_t @ P_ee.unsqueeze(0)
        P_cam_all = T_base_to_cam.unsqueeze(0) @ P_world_all
    else:  # background
        T_cam_to_world_0 = T_ee_t[0] @ T_opt
        P_world = T_cam_to_world_0 @ P_cam0.T
        T_cam_to_world_all = T_ee_t @ T_opt.unsqueeze(0)
        T_world_to_cam_all = torch.linalg.inv(T_cam_to_world_all)
        P_cam_all = T_world_to_cam_all @ P_world.unsqueeze(0)

    Z = P_cam_all[:, 2, :].clamp(min=1e-4)
    u_pred = K[0, 0] * P_cam_all[:, 0, :] / Z + K[0, 2]
    v_pred = K[1, 1] * P_cam_all[:, 1, :] / Z + K[1, 2]
    pred = torch.stack([u_pred, v_pred], dim=-1)

    # L1 per-point error
    pixel_err = (pred - targets).abs().sum(dim=-1)  # (T, N)
    valid = vis & (Z > 0.05)

    T_frames, N_tracks = pixel_err.shape
    err_np = pixel_err.detach().cpu().numpy()
    valid_np = valid.detach().cpu().numpy()

    # ── Per-frame mean error ──
    per_frame_err = []
    for t in range(T_frames):
        v = valid_np[t]
        if v.any():
            per_frame_err.append(err_np[t, v].mean())
        else:
            per_frame_err.append(np.nan)
    per_frame_err = np.array(per_frame_err)

    # ── Per-track mean error ──
    per_track_err = []
    for n in range(N_tracks):
        v = valid_np[:, n]
        if v.any():
            per_track_err.append(err_np[v, n].mean())
        else:
            per_track_err.append(np.nan)
    per_track_err = np.array(per_track_err)

    overall = err_np[valid_np].mean() if valid_np.any() else float('nan')

    print(f"\n{'='*60}")
    print(f"📷 [{cam_id}] scheme={scheme}  N={N_tracks}  T={T_frames}")
    print(f"   Overall L1: {overall:.2f} px")
    print(f"   t=0 error:  {per_frame_err[0]:.4f} px (should be ~0)")
    print(f"   Per-frame:  min={np.nanmin(per_frame_err):.2f}  "
          f"max={np.nanmax(per_frame_err):.2f}  "
          f"median={np.nanmedian(per_frame_err):.2f}")
    print(f"   Per-track:  min={np.nanmin(per_track_err):.2f}  "
          f"max={np.nanmax(per_track_err):.2f}  "
          f"median={np.nanmedian(per_track_err):.2f}")

    # ── Histograms ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(per_frame_err)
    axes[0].set_xlabel("Frame t")
    axes[0].set_ylabel("Mean L1 px error")
    axes[0].set_title(f"[{cam_id}] Per-frame error")
    axes[0].axhline(y=overall, color='r', linestyle='--', label=f'mean={overall:.1f}')
    axes[0].legend()

    axes[1].hist(per_track_err[~np.isnan(per_track_err)], bins=30)
    axes[1].set_xlabel("Mean L1 px error per track")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"[{cam_id}] Per-track error distribution")
    axes[1].axvline(x=overall, color='r', linestyle='--')

    # Error heatmap (T × N)
    im = axes[2].imshow(err_np * valid_np, aspect='auto', cmap='hot',
                         vmin=0, vmax=min(50, np.nanmax(err_np)))
    axes[2].set_xlabel("Track index")
    axes[2].set_ylabel("Frame t")
    axes[2].set_title(f"[{cam_id}] Error heatmap (T×N)")
    plt.colorbar(im, ax=axes[2], label='px')

    plt.tight_layout()
    plt.show()

    # ── Top-5 worst tracks ──
    worst_idx = np.argsort(per_track_err)[::-1][:5]
    print(f"   Top-5 worst tracks:")
    for i, idx in enumerate(worst_idx):
        u0 = targets[0, idx, 0].item()
        v0 = targets[0, idx, 1].item()
        print(f"     #{i+1}: track {idx} (u0={u0:.0f}, v0={v0:.0f}) "
              f"mean_err={per_track_err[idx]:.1f} px")

In [ ]:
# @title 2B. ☁️ LOAD extrinsics from GCS bucket (skip extrinsics computation)

GCS_EXT = "gs://dm-tapnet/mv-tap/droid/extrinsics"
local_ext_cache = f"/content/droid_extrinsics_cache/{episode_id}"
os.makedirs(local_ext_cache, exist_ok=True)

scene_state = {}
for cam in scene_constants['camera']:
    cam_dir = os.path.join(local_ext_cache, cam)
    os.makedirs(cam_dir, exist_ok=True)

    gcs_path = f"{GCS_EXT}/{episode_id}/{cam}/extrinsics.json"
    local_path = os.path.join(cam_dir, "extrinsics.json")
    os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")

    if os.path.exists(local_path):
        with open(local_path) as f:
            ext_data = json.load(f)
        scene_state[cam] = {
            'base_extrinsic': np.array(ext_data['base_extrinsic'], dtype=np.float32),
            'extrinsics': np.array(ext_data['extrinsics'], dtype=np.float32),
            'is_wrist': ext_data.get('is_wrist', False),
        }
        flag = "🦿" if scene_state[cam]['is_wrist'] else "🎥"
        print(f"  ✅ [{cam}] {flag} Shape: {scene_state[cam]['extrinsics'].shape}")
    else:
        print(f"  ⚠️ [{cam}] missing")

# --- Evaluate loaded extrinsics ---
from compute_extrinsics import evaluate_extrinsics, print_metrics, prepare_track_anchors
from core.physics import PyBulletRenderer
if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

base = evaluate_extrinsics(scene_constants, scene_state, device)
print_metrics(base, "Final Extrinsics (GCS)")
for method, tr in tracking_results.items():
    try:
        for cid, d in tr.items():
            scene_constants['camera'][cid]['tracks_2d'] = d['tracks_2d']
            scene_constants['camera'][cid]['vis_2d'] = d['vis_2d']
        anchors = prepare_track_anchors(
            scene_constants, scene_state, pb_renderer, device)
        m = evaluate_extrinsics(scene_constants, scene_state, device,
                                track_anchors=anchors)
        print(f"  🎯 Track reproj ({method:>10s}): "
              f"{m['track_reproj_mean_px']:.2f} px")
    except Exception as e:
        print(f"  ⚠️ Track reproj ({method:>10s}): skipped — {e}")

print("✅ Extrinsics LOADED from GCS")

In [ ]:
# @title 2a. Camera axes overlay
from utils.visualization import render_cross_camera_axes

try:
    axes_frames = render_cross_camera_axes(scene_constants, scene_state, max_frames=30)
    if axes_frames:
        media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")
except Exception as e:
    print(f"Axes visualization skipped: {e}")

In [ ]:
# @title 2b. Robot segmentation video
from utils.visualization import render_segmentation_video
from core.physics import PyBulletRenderer

if 'pb_renderer' not in dir():
    pb_renderer = PyBulletRenderer()

try:
    seg_frames = render_segmentation_video(scene_constants, scene_state, pb_renderer, max_frames=30)
    if seg_frames:
        media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")
except Exception as e:
    print(f"Segmentation visualization skipped: {e}")

In [ ]:
# @title 2c. Fused 3D point cloud
from utils.visualization import render_fused_point_cloud

try:
    render_fused_point_cloud(scene_constants, scene_state, frame_idx=0, height=600, width=1000)
except Exception as e:
    print(f"Fused point cloud skipped: {e}")

In [ ]:
# @title 2d. 4D cinematic orbit
from utils.visualization import render_cinematic_4d_orbit

try:
    orbit_frames = render_cinematic_4d_orbit(scene_constants, scene_state, max_frames=30)
    media.show_video(orbit_frames, fps=10, title="4D Orbit")
except Exception as e:
    print(f"4D Orbit skipped: {e}")

---
## 3. Stage 3: Tracking

⚠️ **Run ONE of the two cells below** (Compute OR Load).

In [ ]:
# @title 3A. 🚧 COMPUTE tracks from scratch (CoTracker + URDF + multi-view fusion)

from compute_tracks import (
    init_tracking_models,
    phase1_extract_2d_tracks, phase2_lift_and_filter,
    phase3_3d_dedup, phase4_cross_view_completion,
    phase5_median_3d_fusion, export_tracks,
)

# Init models (only first time)
if 'cotracker_model' not in dir():
    cotracker_model = init_tracking_models()

camera_ids = list(scene_constants['camera'].keys())

# Phase 1-5
scene_constants = phase1_extract_2d_tracks(cotracker_model, scene_constants, device)
per_cam_env = phase2_lift_and_filter(scene_constants, scene_state, pb_renderer)
unified_pts_3d, unified_to_cam, N_unified = phase3_3d_dedup(per_cam_env, camera_ids)
per_cam_tracks, per_cam_vis = phase4_cross_view_completion(
    cotracker_model, scene_constants, scene_state,
    per_cam_env, unified_pts_3d, unified_to_cam, N_unified, device)
(final_traj_3d, final_vis_global,
 final_per_cam_tracks, final_per_cam_vis, n_survived) = phase5_median_3d_fusion(
    scene_constants, scene_state, per_cam_tracks, per_cam_vis, N_unified)

export_tracks(scene_constants, scene_state,
              final_traj_3d, final_vis_global,
              final_per_cam_tracks, final_per_cam_vis)
print(f"✅ Stage 3 COMPUTE complete: {final_traj_3d.shape[1]} points")

In [ ]:
# @title 3B. ☁️ LOAD tracks from GCS bucket (skip Stage 3 computation)

GCS_TRACKS = "gs://dm-tapnet/mv-tap/droid/tracks"
local_tracks_cache = f"/content/droid_tracks_cache/{episode_id}"
os.makedirs(local_tracks_cache, exist_ok=True)

# Download global 3D tracks + metadata
for fname in ["tracks_3d.npz", "track_metadata.npz"]:
    gcs_path = f"{GCS_TRACKS}/{episode_id}/{fname}"
    local_path = os.path.join(local_tracks_cache, fname)
    ret = os.system(f"gsutil cp '{gcs_path}' '{local_path}' > /dev/null 2>&1")
    if ret == 0:
        print(f"  ✅ {fname}")
    else:
        print(f"  ⚠️  {fname} not found (skipping)")

# Load global tracks
data_3d = np.load(os.path.join(local_tracks_cache, "tracks_3d.npz"))
final_traj_3d = data_3d["traj_3d"]       # (T, N, 3)
final_vis_global = data_3d["vis_global"]  # (T, N)

# Load env/robot split metadata
meta_path = os.path.join(local_tracks_cache, "track_metadata.npz")
if os.path.exists(meta_path):
    meta = np.load(meta_path)
    n_env   = int(meta["n_env"])
    n_robot = int(meta["n_robot"])
else:
    n_env, n_robot = final_traj_3d.shape[1], 0

T, N, _ = final_traj_3d.shape
print(f"  ✅ tracks_3d: {T} frames × {N} points ({n_env} env + {n_robot} robot)")

# Download per-camera 2D tracks
final_per_cam_tracks = {}
final_per_cam_vis = {}

for cam_id in scene_constants["camera"]:
    cam_cache = os.path.join(local_tracks_cache, cam_id)
    os.makedirs(cam_cache, exist_ok=True)

    gcs_cam = f"{GCS_TRACKS}/{episode_id}/{cam_id}"
    for fname in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"]:
        os.system(f"gsutil cp '{gcs_cam}/{fname}' '{cam_cache}/' > /dev/null 2>&1")

    t2d_path = os.path.join(cam_cache, "tracks_2d.npz")
    if os.path.exists(t2d_path):
        d = np.load(t2d_path)
        final_per_cam_tracks[cam_id] = d["traj_2d"]   # (T, N, 2)
        final_per_cam_vis[cam_id]    = d["vis_2d"]     # (T, N)
        print(f"  ✅ Camera [{cam_id}]: 2D tracks loaded")
    else:
        print(f"  ⚠️  Camera [{cam_id}]: tracks_2d.npz not found")

print(f"\n✅ Stage 3 LOADED from GCS — {len(final_per_cam_tracks)} cameras")

In [ ]:
# @title 3. Visualize tracking results
from utils.visualization import render_2d_tracking_video

camera_ids = list(scene_constants['camera'].keys())

all_frames = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    if cam_id in final_per_cam_tracks:
        tracks = final_per_cam_tracks[cam_id]
        vis = final_per_cam_vis[cam_id]
        frames = render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            tgt_size=(256, 456), linewidth=1, max_frames=30)
        all_frames.append(np.array(frames))

if all_frames:
    combined = np.concatenate(all_frames, axis=2)  # horizontal concat
    media.show_video(combined, fps=10, title="Tracks [all cameras]")

---
## Summary

```
per stage:
  A. 🚧 COMPUTE — run from scratch (debug single episode)
  B. ☁️ LOAD    — pull from GCS   (skip to later stages)

droid/
├── compute_depth.py          → Stage 1
├── compute_extrinsics.py     → Stage 2
├── compute_tracks.py         → Stage 3
├── core/                     → Shared modules
│   ├── geometry.py, io.py, depth.py, physics.py, tracking.py
└── utils/visualization.py    → All viz helpers
```

Typical debug workflow:
1. Run Stage 1 once → `run_parallel.sh` on GCP → results on GCS
2. Open notebook → **1B. Load** depth from GCS → 🚧 debug Stage 2
3. Stage 2 works → `run_parallel.sh --stage 2` → results on GCS
4. Open notebook → **1B + 2B. Load** both → 🚧 debug Stage 3